In [3]:
# ============================================================================
# PREREQUISITE SETUP - RUN THIS FIRST
# ============================================================================

# Install required libraries
!pip install xgboost lightgbm catboost shap scikit-learn matplotlib seaborn plotly -q

# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve)
from sklearn.preprocessing import label_binarize
import joblib
import pickle
import json
from datetime import datetime
import time

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# DATA LOADING AND PREPARATION
# ============================================================================

# Load your dataset (update the path)
df = pd.read_csv('/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Data set/processed data set new.csv')  # UPDATE THIS PATH

# ============================================================================
# MODIFICATION START: Sample 1000 rows
# ============================================================================
TARGET_ROWS = 1000
if len(df) > TARGET_ROWS:

    try:
        df_sampled = df.groupby('job_title', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(TARGET_ROWS * len(x) / len(df))), random_state=42)
        )
        # Ensure we don't exceed TARGET_ROWS due to rounding
        df_sampled = df_sampled.head(TARGET_ROWS)

        # If stratified sampling didn't get exactly 1000 rows or failed, fall back to simple sample
        if len(df_sampled) != TARGET_ROWS:
            df_sampled = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
            print(f" Fell back to simple random sampling for 1000 rows due to class imbalance.")

        df = df_sampled
    except Exception:
        df = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
        print(f"⚠️ Simple random sampling used for 1000 rows.")

print("\n" + "="*70)
print(f"DATASET OVERVIEW (REDUCED TO {len(df)} ROWS)")
print("="*70)
# ============================================================================
# MODIFICATION END
# ============================================================================

# Display basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Define features and target
feature_cols = [
    'search_country_encoded', 'job_level_encoded', 'job_type_onsite',
    'job_title_word_count_scaled', 'skills_word_count_scaled',
    'job_title_char_count_scaled', 'skills_char_count_scaled',
    'skills_list_count_scaled', 'title_tokens_count_scaled',
    'skills_tokens_count_scaled', 'job_level_encoded_scaled',
    'search_country_encoded_scaled',
    'has_skill_communication', 'has_skill_customer_service',
    'has_skill_teamwork', 'has_skill_leadership',
    'has_skill_communication_skills'
] + [f'tfidf_component_{i}' for i in range(20)]

# Prepare X and y
X = df[feature_cols].fillna(0)
y = df['job_title']

# 2. Handle rare classes
class_counts = y.value_counts()
# The threshold of 2 is fine, but for 1000 rows, this will likely group many classes into "Other"
y = y.apply(lambda x: x if class_counts[x] >= 2 else "Other")

# 3. Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Features: {len(feature_cols)}")
print(f" Samples: {len(X)}") # Should be 1000
print(f"\n Target Distribution (after rare class handling):")
print(y.value_counts())


print(f"\n Training samples: {len(X_train)}") # Should be 800 (80% of 1000)
print(f" Test samples: {len(X_test)}")      # Should be 200 (20% of 1000)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Fell back to simple random sampling for 1000 rows due to class imbalance.

DATASET OVERVIEW (REDUCED TO 1000 ROWS)
Shape: (1000, 66)

Columns: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_title_clean', 'company_clean', 'job_skills_clean', 'job_location_clean', 'job_title_word_count', 'skills_word_count', 'job_title_char_count', 'skills_char_count', 'skills_list_count', 'has_skill_communication', 'has_skill_customer_service', 'has_skill_teamwork', 'has_skill_leadership', 'has_skill_communication_skills', 'job_title_tokens', 'job_skills_tokens', 'title_tokens_count', 'skills_tokens_count', 'combined_t

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================================================
# MODEL 2: LIGHTGBM CLASSIFIER
# ============================================================================

print("\n" + "="*70)
print("TRAINING LIGHTGBM CLASSIFIER")
print("="*70)

from lightgbm import LGBMClassifier

model_dir = f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Notebooks/Model_02_LIGHTGBM_Classifier.ipynb'



TRAINING LIGHTGBM CLASSIFIER


In [5]:
# ============================================================================
# PART 1: EDA BEFORE TRAINING
# ============================================================================

print("\n PART 1: PRE-TRAINING EDA")

# 1. Feature Statistics
feature_stats = X_train.describe().T
feature_stats.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 02 - outputs-stat_reports.csv')

plt.figure(figsize=(12, 8))
sns.boxplot(data=X_train[feature_cols[:15]], orient='h', palette='Set2')
plt.title('Feature Distributions (Box Plot - Top 15 Features)', fontsize=14, fontweight='bold')
plt.xlabel('Value', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 02 - outputs-feature.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Class Balance
plt.figure(figsize=(10, 6))
class_counts = y_train.value_counts()
plt.pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
plt.title('Target Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 02 - outputs-class_balance.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. TF-IDF Components Analysis
tfidf_cols = [f'tfidf_component_{i}' for i in range(20)]
tfidf_data = X_train[tfidf_cols]

plt.figure(figsize=(15, 8))
for i, col in enumerate(tfidf_cols[:10], 1):
    plt.subplot(2, 5, i)
    plt.hist(tfidf_data[col], bins=30, color='teal', alpha=0.7, edgecolor='black')
    plt.title(f'Component {i-1}', fontsize=10)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
plt.suptitle('TF-IDF Component Distributions (First 10)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 02 - outputs-Tf_IDF.png', dpi=300, bbox_inches='tight')
plt.close()

print("Pre-training EDA visualizations saved!")


 PART 1: PRE-TRAINING EDA
Pre-training EDA visualizations saved!


In [ ]:
# ============================================================================
# PART 2: HYPERPARAMETER TUNING (3 ITERATIONS)
# ============================================================================

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

NUM_CLASSES = len(label_encoder.classes_)

print("\n" + "="*70)
print("PART 2: HYPERPARAMETER TUNING (3 Iterations) - LightGBM")
print("="*70)
print(f"Encoded {NUM_CLASSES} unique job titles")
print(f" y_train shape: {y_train_encoded.shape}")

# Define parameter grids
param_grids = [
    # Iteration 1: Broad search
    {
        'n_estimators': [100, 200],
        'max_depth': [-1],
        'learning_rate': [0.01, 0.1],
        'num_leaves': [31, 50],
        'subsample': [0.8, 1.0]
    },
    # Iteration 2: Refined search
    {
        'n_estimators': [150, 200, 250],
        'max_depth': [-1],
        'learning_rate': [0.05, 0.1, 0.15],
        'num_leaves': [40, 50, 60],
        'subsample': [0.85, 0.9]
    },
    # Iteration 3: Fine-tuning
    {
        'n_estimators': [200, 250, 300],
        'max_depth': [-1],
        'learning_rate': [0.08, 0.1],
        'num_leaves': [50, 60],
        'subsample': [0.9],
        'colsample_bytree': [0.8, 0.9]
    }
]

tuning_results = []

for iteration, param_grid in enumerate(param_grids, 1):
    print(f"\n{'-'*60}")
    print(f"Iteration {iteration}/3")
    print(f"{'-'*60}")

    # Base model initialization
    base_model = LGBMClassifier(
        random_state=42,
        objective='multiclass',
        num_class=NUM_CLASSES,
        verbose=-1,
        n_jobs=-1
    )

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=2
    )

    start_time = time.time()
    # Fit uses the X_train and the integer-encoded y_train_encoded
    grid_search.fit(X_train, y_train_encoded)
    end_time = time.time()

    tuning_results.append({
        'iteration': iteration,
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_,
        'time_taken': end_time - start_time
    })

    print(f"\nBest Parameters: {grid_search.best_params_}")
    print(f"Best CV Score: {grid_search.best_score_:.4f}")
    print(f"Time Taken: {end_time - start_time:.2f}s")

# Save tuning results
try:
    with open(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 02 - outputs-tunning_results.json', 'w') as f:
        json.dump(tuning_results, f, indent=4)
    print("\nTuning results saved to JSON.")
except Exception as e:
    print(f"\nSkipping saving results: Error writing file: {e}")

# Get best parameters from best iteration
best_iteration = max(tuning_results, key=lambda x: x['best_score'])
best_params = best_iteration['best_params']

print(f"\n🏆 Best Overall Parameters (Iteration {best_iteration['iteration']}):")
print(json.dumps(best_params, indent=4))

# Save the label encoder
try:
    joblib.dump(label_encoder, f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 02 - outputs-label_encoder.pkl')
    print(f"\nLabel encoder saved to {model_dir}/label_encoder.pkl")
except Exception as e:
     print(f"\nSkipping saving label encoder: Error writing file: {e}")


PART 2: HYPERPARAMETER TUNING (3 Iterations) - LightGBM
Encoded 51 unique job titles
 y_train shape: (800,)

------------------------------------------------------------
Iteration 1/3
------------------------------------------------------------
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best Parameters: {'learning_rate': 0.1, 'max_depth': -1, 'n_estimators': 200, 'num_leaves': 31, 'subsample': 0.8}
Best CV Score: 0.8320
Time Taken: 2713.19s

------------------------------------------------------------
Iteration 2/3
------------------------------------------------------------
Fitting 5 folds for each of 54 candidates, totalling 270 fits


In [ ]:
# ============================================================================
# PART 3: CROSS-VALIDATION WITH BEST PARAMETERS
# ============================================================================

print("\nPART 3: CROSS-VALIDATION (5-Fold)")

final_model = LGBMClassifier(**best_params, random_state=42, verbose=-1)

cv_scores = cross_val_score(
    final_model, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=-1
)

print(f"\nCV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

plt.figure(figsize=(10, 6))
plt.plot(range(1, 6), cv_scores, marker='s', linestyle='-', linewidth=2, markersize=10, color='green')
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.fill_between(range(1, 6), cv_scores.mean() - cv_scores.std(),
                 cv_scores.mean() + cv_scores.std(), alpha=0.2, color='green')
plt.xlabel('Fold', fontsize=12, fontweight='bold')
plt.ylabel('F1 Score', fontsize=12, fontweight='bold')
plt.title('LightGBM Cross-Validation Scores', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 02 - outputs-04_cv_scores.png', dpi=300, bbox_inches='tight')
plt.close()



PART 3: CROSS-VALIDATION (5-Fold)

CV Scores: [0.83773973 0.82531997 0.83118557 0.82531997 0.85033046]
Mean CV Score: 0.8340 (+/- 0.0094)


In [ ]:
# ============================================================================
# PART 4: FINAL MODEL TRAINING
# ============================================================================

print("\nPART 4: FINAL MODEL TRAINING")

start_time = time.time()
final_model.fit(X_train, y_train)
train_time = time.time() - start_time

print(f"Model trained in {train_time:.2f}s")

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)

train_metrics = {
    'accuracy': accuracy_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
}

test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
}

print("\nTRAINING SET METRICS:")
for metric, value in train_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")

print("\nTEST SET METRICS:")
for metric, value in test_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")



PART 4: FINAL MODEL TRAINING
Model trained in 6.80s

TRAINING SET METRICS:
  Accuracy: 1.0000
  Precision: 1.0000
  Recall: 1.0000
  F1_score: 1.0000

TEST SET METRICS:
  Accuracy: 0.8800
  Precision: 0.7914
  Recall: 0.8800
  F1_score: 0.8322


In [2]:
# ============================================================================
# PART 5: POST-TRAINING VISUALIZATIONS (REVISED FOR COMPARISONS)
# ============================================================================

print("\nPART 5: POST-TRAINING VISUALIZATIONS (Comparison Focus)")

# Define a consistent save directory for the final visualizations
save_dir = '/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /model training results'
# Make sure this directory exists or use os.makedirs(save_dir, exist_ok=True) if it doesn't

# --- 1. Target Class Distribution Comparison (Train vs Test) ---
plt.figure(figsize=(15, 7))

# Calculate counts
train_counts = y_train.value_counts().sort_index()
test_counts = y_test.value_counts().sort_index()

# Combine data for plotting
all_labels = sorted(set(train_counts.index) | set(test_counts.index))
train_data = [train_counts.get(label, 0) for label in all_labels]
test_data = [test_counts.get(label, 0) for label in all_labels]

x = np.arange(len(all_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(15, 8))
bars1 = ax.bar(x - width/2, train_data, width, label='Train Set', color='lightblue', edgecolor='black')
bars2 = ax.bar(x + width/2, test_data, width, label='Test Set', color='darkblue', edgecolor='black')

ax.set_xlabel('Job Title Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Target Class Distribution: Train vs Test Set', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(all_labels, rotation=90)
ax.legend()
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /model training results/Model 02 - outputs-09_target_distribution_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("Target Distribution Comparison saved!")


# --- 2. Train vs Test Metrics Comparison (Existing, but using save_dir) ---
metrics_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Train': [train_metrics['accuracy'], train_metrics['precision'],
              train_metrics['recall'], train_metrics['f1_score']],
    'Test': [test_metrics['accuracy'], test_metrics['precision'],
             test_metrics['recall'], test_metrics['f1_score']]
})

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_comparison['Metric']))
width = 0.35

bars1 = ax.bar(x - width/2, metrics_comparison['Train'], width, label='Train', color='lightgreen', edgecolor='black')
bars2 = ax.bar(x + width/2, metrics_comparison['Test'], width, label='Test', color='darkgreen', edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('LightGBM: Train vs Test Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_comparison['Metric'])
ax.legend()
ax.set_ylim([0, 1.1])

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /model training results/Model 02 - outputs-08_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("Metrics Comparison saved!")

# --- 3. Confusion Matrix (Existing, using save_dir) ---
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 8))
# You may need to use a truncated list of class labels for the heatmap ticks if there are too many classes (51 in your case)
# For simplicity, keeping the default non-labeled heatmap here.
sns.heatmap(cm, annot=False, fmt='d', cmap='Greens', cbar=True)
plt.title('LightGBM Confusion Matrix (Test Set)', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /model training results/Model 02 - outputs-05_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()
print("Confusion Matrix saved!")

# --- 4. Feature Importance (Existing, using save_dir) ---
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='forestgreen', edgecolor='black')
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('LightGBM Top 20 Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /model training results/Model 02 - outputs-06_feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()

feature_importance.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /model training results/Model 02 - outputs-feature_importance.csv', index=False)
print("Feature Importance saved!")


PART 5: POST-TRAINING VISUALIZATIONS (Comparison Focus)


NameError: name 'plt' is not defined

In [ ]:
# ============================================================================
# PART 6: SAVE MODEL AND REPORTS
# ============================================================================

print("\nPART 6: SAVING MODEL AND REPORTS")

# --- Define the base directory for Model 02 files ---
# This ensures consistency and prevents OSError
model_output_dir = '/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 02'
import os
os.makedirs(model_output_dir, exist_ok=True)


# 1. Save Model
joblib.dump(final_model, f'{model_output_dir}/lightgbm_model.pkl')
# Correct the print statement to reflect the actual file path
print(f"Model saved: {model_output_dir}/lightgbm_model.pkl")


# 2. Save Classification Report
report = classification_report(y_test, y_test_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(f'{model_output_dir}/classification_report.csv')
print(f"Classification report saved")


# 3. Save Results Summary
results_summary = {
    'model_name': 'LightGBM Classifier',
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'hyperparameter_tuning': tuning_results,
    'best_parameters': best_params,
    'cross_validation': {
        'cv_scores': cv_scores.tolist(),
        'mean_cv_score': float(cv_scores.mean()),
        'std_cv_score': float(cv_scores.std())
    },
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'training_time': train_time,
    'feature_importance_top10': feature_importance.head(10).to_dict('records')
}

with open(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 02-results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=4)

print(f"Results summary saved")


# 4. Save Predictions (FIXED PATH)
predictions_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_test_pred,
    'correct': y_test.values == y_test_pred
})
predictions_df.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 02-predictions.csv', index=False)
print(f"Predictions saved")


print("\n" + "-"*70)
print("LIGHTGBM TRAINING COMPLETE!")
print("-"*70)
print(f"All files saved to: {model_output_dir}")
print(f"Test F1 Score: {test_metrics['f1_score']:.4f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")


PART 6: SAVING MODEL AND REPORTS
Model saved: /content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 02/lightgbm_model.pkl
Classification report saved
Results summary saved
Predictions saved

----------------------------------------------------------------------
LIGHTGBM TRAINING COMPLETE!
----------------------------------------------------------------------
All files saved to: /content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Summaries/Model 02
Test F1 Score: 0.8322
Test Accuracy: 0.8800


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive
